In [3]:
from os import listdir
from os.path import isfile, join
import pandas as pd

In [14]:
dds_path = "DDS/"
dds_files = [f for f in listdir(dds_path) if isfile(join(dds_path, f))]
ensembl_to_symbol = pd.read_csv('ensmbl_list.csv', index_col = 0)
ensembl_to_symbol.head()
ensembl_to_symbol_dic = ensembl_to_symbol.to_dict('index')

In [18]:
selected_genes = pd.read_csv("selected_genes_orthologs_c_elegans_pathways.csv", index_col = 0)

In [21]:
genes = selected_genes.index

In [42]:
lfc_gene = {}
for dds in dds_files:
    dds_df = pd.read_csv(dds_path+"/"+dds, index_col=0)
    dds_df = dds_df.dropna()
    dict_gene = {}
    title = dds.replace('RNAseq_abundances_adjusted_combat_inmose_', "").replace("_DDS.csv", "")
    for gene in genes:
        if gene not in lfc_gene:
            lfc_gene[gene]={}
        values = dds_df.loc[dds_df.index == gene]
        if not values.empty:
            pvalue = values['pvalue'].values[0]
            lfc_value = values['log2FoldChange'].values[0]
        else:
            pvalue = 1
            lfc_value = 0
        if pvalue < 0.01:
            lfc_gene[gene][title] = lfc_value
        else:
            lfc_gene[gene][title] = 0


In [ ]:
lfc_gene

In [45]:
selected_genes.head()

,Ensembl,Shap/tstat,Source,LFC,Best,gorilla_gorilla,saccharomyces_cerevisiae,caenorhabditis_elegans,mus_musculus,pongo_abelii,human,Pathways,n_Pathways,immune_gene
Symbol,,,,,,,,,,,,,,
NT5C2,ENSG00000076685.18,14.135628,Catboost 07,NaN,1.0,,,Y71H10B.1,Nt5c2,NT5C2,NT5C2,['GOBP_RIBONUCLEOSIDE_MONOPHOSPHATE_CATABOLIC_...,57,NaN
ALDOA,ENSG00000285043.1,7.641741,Catboost 07,NaN,4.0,,,aldo-1,Aldoart1,,ALDOA,['GOBP_RIBONUCLEOSIDE_DIPHOSPHATE_METABOLIC_PR...,103,immune
BLCAP,ENSG00000166619.12,3.897527,Catboost 07,NaN,8.0,,,Y73E7A.6,Blcap,BLCAP,BLCAP,"['GOBP_APOPTOTIC_PROCESS', 'GOBP_CELLULAR_COMP...",6,NaN
FEZ2,ENSG00000171055.14,2.825766,Catboost 07,NaN,9.0,,,unc-76,Fez2,FEZ2,FEZ2,"['GOBP_TAXIS', 'GOBP_AUTOPHAGOSOME_ORGANIZATIO...",35,NaN
SLC16A3,ENSG00000141526.16,1.558696,Catboost 07,NaN,13.0,,MCH5,mct-3,Slc16a3,SLC16A3,SLC16A3,"['GOCC_POSTSYNAPTIC_MEMBRANE', 'REACTOME_THE_C...",67,NaN


In [ ]:
for key in next(iter(lfc_gene.values())).keys():  # Get all possible sub-keys (column names)
    selected_genes[key] = selected_genes.index.map(lambda gene: lfc_gene.get(gene, {}).get(key, None))
selected_genes

In [48]:
selected_genes.to_csv("Selected_genes_orthologs_pathway_lfc.csv")